# Modelo de Classificação de Barreiras de Acessibilidade

## 1. Objetivo

Este notebook apresenta o desenvolvimento de um modelo de Machine Learning responsável por classificar automaticamente relatos de problemas relacionados à acessibilidade urbana.

A partir da descrição textual de uma ocorrência, o modelo identifica a categoria de barreira correspondente, auxiliando na organização e classificação dos registros recebidos pela plataforma AcessoBairro.

O problema foi tratado como uma tarefa de classificação supervisionada, na qual o algoritmo aprende a relacionar descrições textuais às categorias previamente definidas no conjunto de dados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 2. Carregamento e conhecimento dos dados

O conjunto de dados utilizado neste modelo está armazenado no arquivo `dataset_barreiras.csv`.

O dataset possui 500 registros e duas colunas principais:

- `descricao`: texto contendo o relato de uma barreira de acessibilidade.
- `categoria`: classificação correspondente ao problema descrito.

As categorias utilizadas são:

- `CALCADA_DANIFICADA`
- `RAMPA_AUSENTE`
- `PASSAGEM_OBSTRUIDA`
- `RAMPA_INADEQUADA`
- `TRAVESSIA_INADEQUADA`

Os dados são sintéticos e foram organizados para representar diferentes situações de problemas de acessibilidade urbana. Cada categoria possui 100 registros.

In [ ]:
df = pd.read_csv(
    "../datasets/dataset_barreiras.csv",
    sep=";",
    encoding="utf-8-sig"
)

df.head()

## 3. Análise exploratória dos dados

Nesta etapa, são analisadas as características gerais do conjunto de dados.

São verificadas as dimensões do dataset, os tipos de dados, a existência de valores ausentes e registros duplicados, além da distribuição das categorias.

Também é realizada uma análise do tamanho das descrições textuais, permitindo observar a quantidade de palavras presente nos relatos.

A visualização da distribuição das categorias ajuda a identificar possíveis desequilíbrios entre as classes utilizadas no treinamento.

In [ ]:
print("Dimensões:", df.shape)

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nRegistros duplicados:")
print(df.duplicated().sum())

print("\nDistribuição das categorias:")
print(df["categoria"].value_counts())

### 3.1 Distribuição das Categorias

O gráfico apresenta a quantidade de registros existentes em cada categoria de barreira de acessibilidade.

Essa visualização permite verificar o equilíbrio entre as classes utilizadas no treinamento do modelo.

In [ ]:
df["categoria"].value_counts().plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Distribuição das categorias de barreiras")
plt.xlabel("Categoria")
plt.ylabel("Quantidade de registros")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 3.2 Análise do Tamanho dos Relatos

Nesta etapa, é calculada a quantidade de palavras presentes em cada descrição.

Em seguida, são apresentadas medidas estatísticas e um histograma para visualizar a distribuição do tamanho dos relatos.

Essa análise permite compreender melhor as características dos textos utilizados pelo classificador.

In [ ]:
df["tamanho_texto"] = df["descricao"].apply(
    lambda texto: len(texto.split())
)

print(df["tamanho_texto"].describe())

df["tamanho_texto"].plot(
    kind="hist",
    bins=15,
    figsize=(9, 5)
)

plt.title("Distribuição do tamanho dos relatos")
plt.xlabel("Quantidade de palavras")
plt.ylabel("Frequência")
plt.show()

## 4. Preparação dos Dados

Nesta etapa, são definidas as variáveis utilizadas pelo modelo:

- **X:** contém as descrições textuais das ocorrências.
- **y:** contém as categorias correspondentes a cada descrição.

O dataset é dividido em dois conjuntos:

- 80% dos registros para treinamento.
- 20% dos registros para teste.

A divisão utiliza estratificação para preservar a proporção das categorias nos dois conjuntos, além de um estado aleatório fixo para permitir a reprodução dos resultados.

In [ ]:
X = df["descricao"]
y = df["categoria"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", len(X_train))
print("Teste:", len(X_test))

print("\nCategorias no treinamento:")
print(y_train.value_counts())

print("\nCategorias no teste:")
print(y_test.value_counts())

## 5. Vetorização dos Textos com TF-IDF

Para que o algoritmo consiga trabalhar com os relatos textuais, é necessário transformar as palavras e os caracteres em representações numéricas.

Para isso, são utilizados dois vetorizadores TF-IDF:

- **TF-IDF por palavras:** considera palavras individuais e combinações de até duas palavras.
- **TF-IDF por caracteres:** considera sequências de caracteres de três a cinco posições.

A combinação dessas representações permite aproveitar tanto as palavras presentes nos relatos quanto características menores dos textos.

In [ ]:
tfidf_palavras = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

tfidf_caracteres = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True
)

## 6. Construção do Modelo

O modelo é desenvolvido utilizando um Pipeline, que reúne as etapas de transformação dos textos e classificação.

A estrutura utiliza:

- **FeatureUnion:** combina os resultados dos vetorizadores TF-IDF de palavras e caracteres.
- **LogisticRegression:** algoritmo de classificação responsável por identificar a categoria mais adequada para cada relato.

O uso do Pipeline permite reunir o processamento textual e a classificação em uma única estrutura.

In [ ]:
modelo = Pipeline([
    ("vetorizacao", FeatureUnion([
        ("palavras", tfidf_palavras),
        ("caracteres", tfidf_caracteres)
    ])),

    ("classificador", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

## 7. Treinamento do Modelo

Nesta etapa, o modelo é treinado utilizando os dados separados anteriormente para treinamento.

O algoritmo aprende a relacionar as características numéricas extraídas dos relatos com suas respectivas categorias de barreiras.

Após o treinamento, o modelo está preparado para realizar classificações em novos relatos textuais.

In [ ]:
modelo.fit(X_train, y_train)

print("Treinamento concluído!")

## 8. Realização das Previsões

Após o treinamento, o modelo realiza previsões utilizando o conjunto de teste.

As categorias previstas são armazenadas para serem comparadas com as categorias reais dos registros.

Essa comparação permite avaliar o desempenho do classificador em dados que não foram utilizados durante o treinamento.

In [ ]:
y_pred = modelo.predict(X_test)

print("Previsões realizadas!")

## 9. Avaliação do Modelo

O desempenho do modelo é avaliado por meio de métricas de classificação.

São utilizadas:

- **Acurácia:** indica a proporção de classificações corretas em relação ao total de previsões.
- **Precisão:** indica a proporção de previsões positivas corretas para cada categoria.
- **Recall:** indica a capacidade do modelo de identificar corretamente os registros pertencentes a cada categoria.
- **F1-score:** representa o equilíbrio entre precisão e recall.

Também é apresentado um relatório de classificação com os resultados individuais de cada categoria.

In [ ]:
print("Acurácia:")
print(accuracy_score(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

### 9.1 Matriz de Confusão

A matriz de confusão apresenta a comparação entre as categorias reais e as categorias previstas pelo modelo.

Ela permite visualizar a quantidade de classificações corretas e os possíveis erros cometidos pelo classificador em cada categoria.

In [ ]:
matriz = confusion_matrix(
    y_test,
    y_pred,
    labels=modelo.classes_
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=modelo.classes_
)

disp.plot(
    cmap="Blues",
    xticks_rotation=45
)

plt.title("Matriz de Confusão - Classificação de Barreiras")
plt.tight_layout()
plt.show()

## 10. Exportação do Modelo

Após o treinamento e a avaliação, o modelo é exportado utilizando a biblioteca Joblib.

O arquivo gerado é `modelo_barreiras.joblib`, armazenado na pasta `api`.

Esse arquivo permite reutilizar o modelo posteriormente na API Flask, sem a necessidade de realizar todo o processo de treinamento novamente.

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    modelo,
    "../api/modelo_barreiras.joblib"
)

print("Modelo salvo com sucesso!")

## 11. Conclusão

O desenvolvimento deste modelo permitiu construir um classificador capaz de analisar relatos textuais e identificar cinco categorias de barreiras de acessibilidade urbana.

O modelo utiliza técnicas de vetorização TF-IDF combinadas com o algoritmo Logistic Regression.

Nos testes realizados com o conjunto de dados utilizado, o modelo obteve acurácia de 1,00 (100%).

Entretanto, como o dataset utilizado possui dados sintéticos, esse resultado não garante o mesmo desempenho em relatos reais. A aplicação em cenários reais exige testes e validações com dados representativos de ocorrências urbanas.